In [42]:
from Team import Team
import numpy as np
import pandas as pd
import os
from datetime import datetime

# Globals
curr_year = datetime.now().year

def preprocess_data():

    def preprocess_team_stats(df):
        def get_rolling_df(df):
            numeric_cols = ['Home_Score', 'Away_Score', 'Offense_Total_Yrds', 'Offense_Pass_Yrds', 'Offense_Rush_Yrds',
                            'Turnovers_Lost', 'Total_Yrds_Allowed', 'Pass_Yrds_Allowed',
                            'Rush_Yrds_Allowed', 'Turnovers_Gained', 'Offense_Expected_Points',
                            'Defense_Expected_Points', 'Spteams_Expected_Points', 'Offensive_1sts']
        
            team_stats = df.copy()
        
            team_stats[numeric_cols] = team_stats[numeric_cols].apply(pd.to_numeric, errors='coerce')
        
            team_stats = team_stats.sort_values(['Home_Team', 'Year'])
        
            rolling_stats = team_stats.copy()
        
            for col in numeric_cols:
                rolling_stats[col] = (
                    team_stats
                    .groupby(['Home_Team', 'Year'])[col]
                    .transform(lambda x: x.rolling(window=4, min_periods=1).mean().shift(1))
                )
        
            first_week_mask = team_stats.duplicated(subset=['Home_Team', 'Year'], keep='first') == False
            rolling_stats.loc[first_week_mask, numeric_cols] = pd.NA
        
            team_stats[numeric_cols] = rolling_stats[numeric_cols]
        
            return team_stats
            
        team_list = df['Home_Team'].unique().tolist()
        teams_dict = {}
        
        for team in team_list:
            try:
                try_year = curr_year

                while True:
                    json_path = f'../../data/team_jsons/{team}_2002_{try_year}.json'
                    
                    if os.path.exists(json_path):
                        team_df = pd.read_json(json_path)
                        break  # file found, break out of loop
                    else:
                        try_year -= 1
                        if try_year < 2002:
                            raise FileNotFoundError(f"JSON for {team} does not exist or can't be found")
        
                team_obj = Team(team_df, team)
                cleaned_df = team_obj.assign_year().arrange_team_json().get_df()
                teams_dict[team] = cleaned_df
                
            except Exception as e:
                print(f"Error processing {team} during JSON reading process: {e}")
                continue

        all_team_stats = pd.DataFrame()

        for team in team_list:
            df = teams_dict[team]
            df = get_rolling_df(df)
            all_team_stats = pd.concat([all_team_stats, df], axis=0, ignore_index=True)

        return all_team_stats

    def preprocess_nfl_games(df):
        def change_old_teams(df, team):
            exclude = {"AFC", "NFC", "RIC", "CTR", "IRV"}
            df = df[~df[team].isin(exclude)].copy()
        
            replacements = {
                'STL': 'LAR', 'OAK': 'LV', 'SD': 'LAC', 'CLT': 'IND', 'OTI': 'TEN',
                'NOR': 'NO', 'WAS': 'WSH', 'RAM': 'LAR', 'TAM': 'TB', 'RAV': 'BAL',
                'GNB': 'GB', 'NWE': 'NE', 'SFO': 'SF', 'KAN': 'KC', 'CRD': 'ARI',
                'HTX': 'HOU', 'SDG': 'LAC', 'RAI': 'LV'
            }
            df[team] = df[team].replace(replacements)

            return df

        def adjust_record(record, team_type, winner):
            overall_summary = next((item['summary'] for item in record if item['name'] == 'overall'), None)
            home_summary = next((item['summary'] for item in record if item['name'] == 'Home'), None)
            road_summary = next((item['summary'] for item in record if item['name'] == 'Road'), None)
        
            def parse_summary(summary):
                if summary and '-' in summary:
                    try:
                        wins, losses = map(int, summary.split('-'))
                        return wins, losses
                    except Exception as e:
                        return None, None
                else:
                    return None, None
        
            overall_wins, overall_losses = parse_summary(overall_summary)
            home_wins, home_losses = parse_summary(home_summary)
            road_wins, road_losses = parse_summary(road_summary)
        
            if team_type == "home":
                if winner:  # Home team won
                    if home_wins is not None:
                        home_wins -= 1  # Don't count this win for home team
                    if overall_wins is not None:
                        overall_wins -= 1  # Don't count this win for overall record
                else:  # Away team won
                    if home_losses is not None:
                        home_losses -= 1  # Home team lost, so adjust their loss count
                    if overall_losses is not None:
                        overall_losses -= 1  # Don't count this loss for overall record
            else:  # away team
                if winner:  # Away team won
                    if road_wins is not None:
                        road_wins -= 1  # Don't count this win for away team
                    if overall_wins is not None:
                        overall_wins -= 1  # Don't count this win for overall record
                else:  # Home team won
                    if road_losses is not None:
                        road_losses -= 1  # Away team lost, so adjust their loss count
                    if overall_losses is not None:
                        overall_losses -= 1  # Don't count this loss for overall record
        
            return overall_wins, overall_losses, home_wins, home_losses, road_wins, road_losses

        included_cols = [
            "game_id", "season", "season_type", "week", "venue_id", "venue_indoor", "neutral_site",
            "home_id", "home_name", "home_abbreviation", "home_score", "home_winner",
            "home_records", "home_linescores",
            "away_id", "away_name", "away_abbreviation", "away_score", "away_winner",
            "away_records", "away_linescores",
            "broadcast_market", "broadcast_name", "status_type_description"
        ]
        
        nfl_df = df[included_cols]
        nfl_df = nfl_df.copy()
        nfl_df["broadcast_name"] = nfl_df["broadcast_name"].replace("", "local")
        nfl_df["broadcast_market"] = nfl_df["broadcast_market"].replace("", "local")
        nfl_df[['overall_wins(home)', 'overall_losses(home)', 'home_wins(home)', 'home_losses(home)', 'road_wins(home)',
                'road_losses(home)']] = nfl_df.apply(
            lambda row: pd.Series(adjust_record(row['home_records'], "home", row['home_winner'])), axis=1
        )
        nfl_df[['overall_wins(away)', 'overall_losses(away)', 'home_wins(away)', 'home_losses(away)', 'road_wins(away)',
                'road_losses(away)']] = nfl_df.apply(
            lambda row: pd.Series(adjust_record(row['away_records'], "away", row['away_winner'])), axis=1
        )
        nfl_df = nfl_df[nfl_df['season_type'] != 1]
        nfl_df.rename(
            columns={'home_abbreviation': 'Home_Team', 'away_abbreviation': 'Away_Team', 'season': 'Year', 'week': 'Week'},
            inplace=True)
        
        # *Some older team names are different* This is to change them
        home_away_list = ['Home', 'Away']

        for item in home_away_list:
            nfl_df = change_old_teams(nfl_df, f'{item}_Team')
        
        nfl_df = nfl_df[nfl_df['Year'] >= 2002]
        
        # Don't really care for Postponed status games
        nfl_df = nfl_df[nfl_df['status_type_description'] != 'Postponed']
        
        # Ensure the dataframe is sorted by year and week to process games chronologically
        nfl_df = nfl_df.sort_values(by=['Year', 'Week']).reset_index(drop=True)

        return nfl_df

    def merge_games_and_teams(df, to_merge):
        numeric_cols = ['Home_Score', 'Away_Score', 'Offense_Total_Yrds', 'Offense_Pass_Yrds', 'Offense_Rush_Yrds',
                    'Turnovers_Lost', 'Total_Yrds_Allowed', 'Pass_Yrds_Allowed',
                    'Rush_Yrds_Allowed', 'Turnovers_Gained', 'Offense_Expected_Points',
                    'Defense_Expected_Points', 'Spteams_Expected_Points']

        df = df.merge(to_merge[['Home_Team', 'Year', 'Week', 'season_type'] + numeric_cols],
                      left_on=['Home_Team', 'Year', 'Week', 'season_type'],
                      right_on=['Home_Team', 'Year', 'Week', 'season_type'],
                      how='left', suffixes=('_home_stats', '')).copy()
    
        df = df.merge(to_merge[['Home_Team', 'Year', 'Week', 'season_type'] + numeric_cols],
                      left_on=['Away_Team', 'Year', 'Week', 'season_type'],
                      right_on=['Home_Team', 'Year', 'Week', 'season_type'],
                      how='left', suffixes=('', '_away_stats')).copy()
    
        return df

    main_df = pd.read_json(f'../../data/nfl_games/nfl_2002_{curr_year}.json')
    nfl_df = preprocess_nfl_games(main_df)
    all_team_stats = preprocess_team_stats(nfl_df)
    main_df = merge_games_and_teams(nfl_df, all_team_stats)
    main_df.to_json(f'../../data/preprocessed_data/preprocessed_2002_{curr_year}.json')
    return main_df

,game_id,Year,season_type,Week,venue_id,venue_indoor,neutral_site,home_id,home_name,Home_Team,...,Offense_Pass_Yrds_away_stats,Offense_Rush_Yrds_away_stats,Turnovers_Lost_away_stats,Total_Yrds_Allowed_away_stats,Pass_Yrds_Allowed_away_stats,Rush_Yrds_Allowed_away_stats,Turnovers_Gained_away_stats,Offense_Expected_Points_away_stats,Defense_Expected_Points_away_stats,Spteams_Expected_Points_away_stats
20,220915001,2002,2,2.0,NaN,NaN,False,1,Falcons,ATL,...,288.0,80.0,2.0,368.0,228.0,140.0,3.0,6.37,-1.20,1.01
21,220915005,2002,2,2.0,NaN,NaN,False,5,Browns,CLE,...,167.0,36.0,1.0,401.0,160.0,241.0,NaN,-7.00,-22.74,3.17
22,220915006,2002,2,2.0,NaN,NaN,False,6,Cowboys,DAL,...,267.0,61.0,4.0,261.0,181.0,80.0,3.0,3.00,8.59,-8.80
23,220915011,2002,2,2.0,NaN,NaN,False,11,Colts,IND,...,207.0,182.0,NaN,257.0,206.0,51.0,2.0,21.92,4.00,3.09
24,220915012,2002,2,2.0,NaN,NaN,False,12,Chiefs,KC,...,225.0,118.0,2.0,307.0,203.0,104.0,2.0,5.80,-4.46,-5.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6779,401772916,2025,2,17.0,3948.0,0.0,False,15,Dolphins,MIA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6780,401772808,2025,2,17.0,3839.0,0.0,False,20,Jets,NYJ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6781,401772917,2025,2,17.0,3883.0,0.0,False,2,Bills,BUF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6782,401772935,2025,2,17.0,4738.0,0.0,False,25,49ers,SF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
included_cols = [
    "game_id", "season", "season_type", "week", "venue_id", "venue_indoor", "neutral_site",
    "home_id", "home_name", "home_abbreviation", "home_score", "home_winner",
    "home_records", "home_linescores",
    "away_id", "away_name", "away_abbreviation", "away_score", "away_winner",
    "away_records", "away_linescores",
    "broadcast_market", "broadcast_name", "status_type_description"
]

nfl_df = main_df[included_cols]
nfl_df = nfl_df.copy()
nfl_df["broadcast_name"] = nfl_df["broadcast_name"].replace("", "local")
nfl_df["broadcast_market"] = nfl_df["broadcast_market"].replace("", "local")
nfl_df[['overall_wins(home)', 'overall_losses(home)', 'home_wins(home)', 'home_losses(home)', 'road_wins(home)',
        'road_losses(home)']] = nfl_df.apply(
    lambda row: pd.Series(adjust_record(row['home_records'], "home", row['home_winner'])), axis=1
)
nfl_df[['overall_wins(away)', 'overall_losses(away)', 'home_wins(away)', 'home_losses(away)', 'road_wins(away)',
        'road_losses(away)']] = nfl_df.apply(
    lambda row: pd.Series(adjust_record(row['away_records'], "away", row['away_winner'])), axis=1
)
nfl_df = nfl_df[nfl_df['season_type'] != 1]
nfl_df.rename(
    columns={'home_abbreviation': 'Home_Team', 'away_abbreviation': 'Away_Team', 'season': 'Year', 'week': 'Week'},
    inplace=True)

# *Some older team names are different* This is to change them
nfl_df = change_old_teams(nfl_df, 'Home_Team')
nfl_df = change_old_teams(nfl_df, 'Away_Team')

nfl_df = nfl_df[(nfl_df['Year'] >= 2002) & (nfl_df['Year'] < 2024)]

# Don't really care for Postponed status games
nfl_df = nfl_df[nfl_df['status_type_description'] != 'Postponed']

# Incorporate Elo Ratings
initial_elo = 1500
team_elos = {}  # Dictionary to store Elo ratings for each team

# Ensure the dataframe is sorted by year and week to process games chronologically
nfl_df = nfl_df.sort_values(by=['Year', 'Week']).reset_index(drop=True)

# Add columns for Elo ratings
nfl_df['elo_home'] = initial_elo
nfl_df['elo_away'] = initial_elo
nfl_df = nfl_df.astype({'elo_home': 'float64', 'elo_away': 'float64'})

# Process each game to assign and update Elo ratings
for idx, row in nfl_df.iterrows():
    home_team = row['Home_Team']
    away_team = row['Away_Team']

    # Get current Elo ratings (or initialize if first game)
    home_elo = team_elos.get(home_team, initial_elo)
    away_elo = team_elos.get(away_team, initial_elo)

    # Assign Elo ratings to the current game (before the game is played)
    nfl_df.at[idx, 'elo_home'] = home_elo
    nfl_df.at[idx, 'elo_away'] = away_elo

    # Update Elo ratings based on game outcome
    new_home_elo, new_away_elo = update_elo(
        home_elo, away_elo, row['home_score'], row['away_score']
    )

    # Store updated Elo ratings for the teams' next games
    team_elos[home_team] = new_home_elo
    team_elos[away_team] = new_away_elo

# Remove ties
nfl_df = nfl_df[nfl_df['home_winner'] != nfl_df['away_winner']]

def merge_dfs(df, to_merge):
    numeric_cols = ['Home_Score', 'Away_Score', 'Offense_Total_Yrds', 'Offense_Pass_Yrds', 'Offense_Rush_Yrds',
                    'Turnovers_Lost', 'Total_Yrds_Allowed', 'Pass_Yrds_Allowed',
                    'Rush_Yrds_Allowed', 'Turnovers_Gained', 'Offense_Expected_Points',
                    'Defense_Expected_Points', 'Spteams_Expected_Points']

    df = df.merge(to_merge[['Home_Team', 'Year', 'Week', 'season_type'] + numeric_cols],
                  left_on=['Home_Team', 'Year', 'Week', 'season_type'],
                  right_on=['Home_Team', 'Year', 'Week', 'season_type'],
                  how='left', suffixes=('_home', ''))

    df = df.merge(to_merge[['Home_Team', 'Year', 'Week', 'season_type'] + numeric_cols],
                  left_on=['Away_Team', 'Year', 'Week', 'season_type'],
                  right_on=['Home_Team', 'Year', 'Week', 'season_type'],
                  how='left', suffixes=('', '_away'))

    return df


# In[29]:


team_list = nfl_df['Home_Team'].unique().tolist()
all_team_stats = pd.DataFrame()

for team in team_list:
    df = arrange_team_json(teams_dict[team], team)
    df = get_rolling_df(df)
    all_team_stats = pd.concat([all_team_stats, df], axis=0, ignore_index=True)


nfl_df = merge_dfs(nfl_df, all_team_stats)
nfl_df['turnover_differential'] = (nfl_df['Turnovers_Gained'] - nfl_df['Turnovers_Lost']) / (
            nfl_df['Turnovers_Gained_away'] - nfl_df['Turnovers_Lost_away']).replace(0, np.nan)
nfl_df

# In[32]:


nfl_df[(nfl_df['Home_Score'].isna()) & (nfl_df['season_type'] == 2)]

# In[33]:


pd.reset_option('display.max_rows')


# In[34]:


def average_terms(df, term1, term2):
    return df[term1] + df[term2] / 2
    

# Home team performance features
home_features = [
    "Home_Score", "Offense_Total_Yrds", "Offense_Pass_Yrds", "Offense_Rush_Yrds",
    "Turnovers_Lost", "Total_Yrds_Allowed", "Pass_Yrds_Allowed", "Rush_Yrds_Allowed",
    "Turnovers_Gained", "Offense_Expected_Points", "Defense_Expected_Points", "Spteams_Expected_Points",
    "turnover_differential"
]

# Away team rolling average features
away_features = [
    "Home_Score_away", "Offense_Total_Yrds_away", "Offense_Pass_Yrds_away", "Offense_Rush_Yrds_away",
    "Turnovers_Lost_away", "Total_Yrds_Allowed_away", "Pass_Yrds_Allowed_away", "Rush_Yrds_Allowed_away",
    "Turnovers_Gained_away", "Offense_Expected_Points_away", "Defense_Expected_Points_away",
    "Spteams_Expected_Points_away"
]

# Create interaction terms (multiply home and away features)
offense_terms = ["Pass", "Rush", "Total"]
home_away_terms = ['', '_away']

for term in offense_terms:
    for status in home_away_terms:
        other_status = ''

        if status == '':
            other_status = '_away'

        nfl_df[f'Expected_{term}_Yrds{status}'] = average_terms(nfl_df, f'Offense_{term}_Yrds{status}',
                                                                f'{term}_Yrds_Allowed{other_status}')

# Create Elo diff
nfl_df['elo_diff'] = nfl_df['elo_home'] - nfl_df['elo_away']

# In[36]:


nfl_df

# In[37]:


"""
selected_features = home_features
selected_features.append("home_winner")
sns.pairplot(nfl_df[selected_features], hue ='home_winner')
plt.show()
"""

# In[38]:


sns.boxplot(x='home_winner', y='elo_diff', data=nfl_df)

# In[39]:


target_col = "home_winner"
all_cols = nfl_df.columns[24:].drop('Home_Team_away').tolist()
print(all_cols)

categorical_cols = []

numerical_cols = all_cols

# Boxplots of all num columns
"""
for col in numerical_cols:
    sns.boxplot(x=X_train[col])
    plt.show()
"""

nfl_df = nfl_df.dropna(subset=[target_col])

num_wins = (nfl_df['home_winner'] == 1).sum()
num_losses = (nfl_df['home_winner'].shape[0] - num_wins)
class_weights = num_losses / num_wins
X = nfl_df[numerical_cols + categorical_cols]
y = nfl_df[target_col].astype(int)